## Gamified Course Journey

**Click the cookies in the game to unlock course zones!**  
Your progress is automatically saved!


In [ ]:
%%html
<!-- UI_RUNNER: Zone unlock panel with buttons | panel: home_zone_journey, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<!-- Styles defined in: _sass/open-coding/forms/home-gamified.scss -->

<div id="zone-unlock-panel">
  <h3>Progress Tracker</h3>
  <p class="zone-subtitle">Total Clicks: <span id="total-clicks">0</span></p>
  
  <button class="zone-btn locked" data-zone="CSSE" disabled>
    <div class="zone-btn-title">CSSE</div>
    <div class="zone-btn-subtitle">Foundation</div>
    <div class="zone-btn-lock">🔒 5 clicks</div>
  </button>

  <button class="zone-btn locked" data-zone="CSP" disabled>
    <div class="zone-btn-title">CSP</div>
    <div class="zone-btn-subtitle">Projects</div>
    <div class="zone-btn-lock">🔒 10 clicks</div>
  </button>

  <button class="zone-btn locked" data-zone="CSA" disabled>
    <div class="zone-btn-title">CSA</div>
    <div class="zone-btn-subtitle">Systems</div>
    <div class="zone-btn-lock">🔒 15 clicks</div>
  </button>

  <button class="zone-btn locked" data-zone="CSH" disabled>
    <div class="zone-btn-title">CSH</div>
    <div class="zone-btn-subtitle">Capstone</div>
    <div class="zone-btn-lock">🔒 20 clicks</div>
  </button>
</div>

<script>
// Global state with localStorage
function loadProgress() {
  const saved = localStorage.getItem('ocsZoneProgress');
  if (saved) {
    try {
      return JSON.parse(saved);
    } catch (e) {
      console.error('Error loading progress:', e);
    }
  }
  return { totalClicks: 0, unlocked: {}, selectedZone: null };
}

function saveProgress() {
  localStorage.setItem('ocsZoneProgress', JSON.stringify({
    totalClicks: window.ocsZoneJourney.totalClicks,
    unlocked: window.ocsZoneJourney.unlocked,
    selectedZone: window.ocsZoneJourney.selectedZone
  }));
}

const savedProgress = loadProgress();
window.ocsZoneJourney = {
  totalClicks: savedProgress.totalClicks || 0,
  unlocked: savedProgress.unlocked || {},
  selectedZone: savedProgress.selectedZone || null
};

const zoneData = {
  CSSE: {
    title: 'CSSE - Software Engineering',
    subtitle: 'Foundation',
    mission: 'CSSE 1,2 prepares students for the AP Computer Science pathway through game development projects.',
    unlockThreshold: 5
  },
  CSP: {
    title: 'CSP - Computer Science Principles',
    subtitle: 'Projects',
    mission: 'Students work on individual and team projects to build computer systems and write algorithms.',
    unlockThreshold: 10
  },
  CSA: {
    title: 'CSA - Computer Science A',
    subtitle: 'Systems',
    mission: 'In-depth course focusing on programming, algorithms, and data structures using Java.',
    unlockThreshold: 15
  },
  CSH: {
    title: 'CSH - Computer Science Honors',
    subtitle: 'Capstone',
    mission: 'Year-long senior thesis emphasizing professional collaboration and real-world solutions.',
    unlockThreshold: 20
  }
};

function updateZoneButtons() {
  const journey = window.ocsZoneJourney;
  const totalClicks = journey.totalClicks;
  
  ['CSSE', 'CSP', 'CSA', 'CSH'].forEach(zone => {
    const data = zoneData[zone];
    const unlocked = totalClicks >= data.unlockThreshold;
    const btn = document.querySelector('.zone-btn[data-zone="' + zone + '"]');
    
    if (!btn) return;
    
    journey.unlocked[zone] = unlocked;
    
    if (unlocked) {
      btn.classList.remove('locked');
      btn.classList.add('unlocked');
      btn.disabled = false;
      const lockDiv = btn.querySelector('.zone-btn-lock');
      if (lockDiv) lockDiv.textContent = '✅';
    } else {
      btn.classList.remove('unlocked');
      btn.classList.add('locked');
      btn.disabled = true;
      const lockDiv = btn.querySelector('.zone-btn-lock');
      const remaining = data.unlockThreshold - totalClicks;
      if (lockDiv) lockDiv.textContent = '🔒 ' + remaining + ' more';
    }
    
    // Update active state
    if (journey.selectedZone === zone) {
      btn.classList.add('active');
    } else {
      btn.classList.remove('active');
    }
  });
  
  // Update zone section visibility
  updateZoneSectionVisibility();
  saveProgress();
}

function updateZoneSectionVisibility() {
  const journey = window.ocsZoneJourney;
  
  ['CSSE', 'CSP', 'CSA', 'CSH'].forEach(zone => {
    const section = document.getElementById('zone-section-' + zone);
    const zoneName = 'zone_' + zone.toLowerCase();
    
    // Find UI_RUNNER and GAME_RUNNER containers for this zone
    const uiContainer = document.querySelector('[id^="uirunner-' + zoneName + '"]');
    const gameContainer = document.querySelector('[id^="gamerunner-' + zoneName + '"]');
    
    const isActive = journey.selectedZone === zone && journey.unlocked[zone];
    
    if (section) {
      section.style.display = isActive ? 'block' : 'none';
    }
    
    // Show/hide zone panels
    if (uiContainer) {
      if (isActive) {
        uiContainer.classList.add('zone-active');
      } else {
        uiContainer.classList.remove('zone-active');
      }
    }
    
    if (gameContainer) {
      if (isActive) {
        gameContainer.classList.add('zone-active');
      } else {
        gameContainer.classList.remove('zone-active');
      }
    }
  });
}

function incrementClicks() {
  window.ocsZoneJourney.totalClicks++;
  document.getElementById('total-clicks').textContent = window.ocsZoneJourney.totalClicks;
  updateZoneButtons();
}

// Handle zone button clicks
document.addEventListener('click', function(e) {
  const btn = e.target.closest('.zone-btn');
  if (!btn || btn.disabled) return;
  
  const zone = btn.dataset.zone;
  window.ocsZoneJourney.selectedZone = zone;
  
  updateZoneButtons();
});

// Listen for cookie clicks from game
window.addEventListener('cookie-click', incrementClicks);

// Initialize display
updateZoneButtons();
</script>

In [ ]:
%%js

// GAME_RUNNER: Journey launcher | hide_edit: true, panel: home_zone_journey, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 320px

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import Clicker from '/assets/js/GameEnginev1.1/essentials/Clicker.js';

class ZoneJourneyGame {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const zone = window.ocsZoneJourney?.currentZone || 'CSSE';
    const unlocked = window.ocsZoneJourney?.unlocked[zone] || false;

    // Zone-specific assets
    const zoneAssets = {
      CSSE: {
        sprites: [path + '/hacks/cookie-clicker/assets/baseCookie.png', 
                  path + '/hacks/cookie-clicker/assets/grandma.png'],
        color: '#667eea',
        speed: [2, 5],
        count: 3
      },
      CSP: {
        sprites: [path + '/hacks/cookie-clicker/assets/baseCookie.png',
                  path + '/hacks/cookie-clicker/assets/grandma.png'],
        color: '#f093fb',
        speed: [2, 5],
        count: 4
      },
      CSA: {
        sprites: [path + '/hacks/cookie-clicker/assets/baseCookie.png',
                  path + '/hacks/cookie-clicker/assets/grandma.png'],
        color: '#4facfe',
        speed: [2, 5],
        count: 5
      },
      CSH: {
        sprites: [path + '/hacks/cookie-clicker/assets/baseCookie.png',
                  path + '/hacks/cookie-clicker/assets/grandma.png'],
        color: '#43e97b',
        speed: [2, 5],
        count: 6
      }
    };

    const cfg = zoneAssets[zone] || zoneAssets.CSSE;

    const spriteData = {
      id: zone + ' Journey Token',
      greeting: unlocked ? "Welcome to " + zone + "! Click to explore!" : "Click to unlock " + zone + "!",
      src: cfg.sprites[0],
      SCALE_FACTOR: 7,
      pixels: { height: 512, width: 512 },
      INIT_POSITION: { x: 0, y: 0 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      up: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      left: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      right: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      hitbox: { widthPercentage: 0.2, heightPercentage: 0.2 },
      walkingArea: { xMin: 0, xMax: width, yMin: 0, yMax: height },
      speed: 3,
      direction: { x: 1, y: 1 },
      interact: function(clicks, objectId) {
        // Dispatch cookie click event to increment global counter
        window.dispatchEvent(new CustomEvent('cookie-click'));
        
        const journey = window.ocsZoneJourney;
        const metric = document.querySelector('[id^="gamerunner-home-gamified-mvp"][id$="-metric"]');
        if (metric) {
          metric.textContent = "Total Clicks: " + journey.totalClicks;
        }
      }
    };

    this.classes = [{
      class: Clicker,
      data: spriteData,
      spawn: {
        count: cfg.count,
        ranges: {
          INIT_POSITION: {
            x: [0, Math.max(0, width - 128)],
            y: [0, Math.max(0, height - 128)]
          },
          speed: cfg.speed
        },
        pickOne: { src: cfg.sprites }
      }
    }];
  }
}

export const gameLevelClasses = [ZoneJourneyGame];
export { GameControl };

In [ ]:
%%html
<!-- UI_RUNNER: CSSE Content | panel: zone_csse, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<!-- Styles defined in: _sass/open-coding/forms/home-gamified.scss -->

<div id="zone-section-CSSE" class="zone-section" style="display: none;">
  <div class="zone-content">
    <h2>🎮 CSSE Zone - Software Engineering Foundation</h2>
    <p><em>Game development focused learning path</em></p>
    
    <h3>CSSE - Software Engineering</h3>
    <p class="zone-subtitle-text">Foundation</p>
    <p class="zone-description">
      CSSE 1,2 prepares students for the AP Computer Science pathway. This course focuses on teaching the JavaScript programming language, object-oriented programming and inheritance, and developing algorithmic thinking skills through game development projects.
    </p>
    <h4>Learning Journey:</h4>
    <ul class="zone-journey-list">
      <li>Game Builder</li>
      <li>JS Fundamentals (Code Runner)</li>
      <li>GitHub Project and VSCode</li>
      <li>Game Objects</li>
      <li>Sprite Animation Studio</li>
      <li>WASD Maze Navigator</li>
    </ul>
  </div>
</div>

In [ ]:
%%js

// GAME_RUNNER: CSSE Zone Game | hide_edit: true, panel: zone_csse, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GamEnvBackground from '/assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1.1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1.1/essentials/Npc.js';
import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';

class GameLevelCSSE {
  constructor(gameEnv) {
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const path = gameEnv.path;

    // CSSE themed background - game development desert
    const bgData = {
      name: 'csse-desert',
      greeting: "Welcome to CSSE! Learn JavaScript through game development!",
      src: path + "/images/gamify/desert.png",
      pixels: { height: 580, width: 1038 }
    };

    // Player - Chill Guy as beginner programmer
    const playerData = {
      id: 'CSSE Student',
      greeting: "Hi! I'm learning JavaScript and game development!",
      src: path + "/images/gamify/chillguy.png",
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: 0, y: height - (height/5) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down: { row: 0, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    // NPC - R2D2 as CSSE guide
    const npcData = {
      id: 'CSSE Guide',
      greeting: "Beep boop! Welcome to game development! Learn JavaScript fundamentals here!",
      src: path + "/images/gamify/r2_idle.png",
      SCALE_FACTOR: 8,
      ANIMATION_RATE: 100,
      pixels: { width: 505, height: 223 },
      INIT_POSITION: { x: width * 1 / 4, y: height * 3 / 4 },
      orientation: { rows: 1, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
      dialogues: [
        "Start with Game Builder to create your first interactive game!",
        "JavaScript is the language of the web and game development!",
        "Learn object-oriented programming through game objects!",
        "WASD controls are fundamental to game navigation!",
        "GitHub and VSCode are essential tools for developers!",
        "Sprite animation brings your games to life!"
      ]
    };

    this.classes = [
      { class: GamEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npcData }
    ];
  }
}

export const gameLevelClasses = [GameLevelCSSE];
export { GameControl };

In [ ]:
%%html
<!-- UI_RUNNER: CSP Content | panel: zone_csp, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<div id="zone-section-CSP" class="zone-section" style="display: none;">
  <div class="zone-content">
    <h2>🐍 CSP Zone - Computer Science Principles</h2>
    <p><em>Project-based learning with algorithms and data structures</em></p>
    
    <h3>CSP - Computer Science Principles</h3>
    <p class="zone-subtitle-text">Projects</p>
    <p class="zone-description">
      Computer Science Principles is designed as a college-level introduction to computer science. Students work on individual and team projects to build computer systems, write algorithms, analyze for correctness, and engage in discussions about solutions.
    </p>
    <h4>Learning Journey:</h4>
    <ul class="zone-journey-list">
      <li>Python Fundamentals</li>
      <li>Algorithm Design & Analysis</li>
      <li>Data Structures Exploration</li>
      <li>Flask Web Applications</li>
      <li>SQL Databases</li>
      <li>Team Project Development</li>

    </ul></div>
  </div>

In [ ]:
%%js

// GAME_RUNNER: CSP Zone Game | hide_edit: true, panel: zone_csp, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GamEnvBackground from '/assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1.1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1.1/essentials/Npc.js';
import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';

class GameLevelCSP {
  constructor(gameEnv) {
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const path = gameEnv.path;

    const bgData = {
      name: 'csp-desert',
      greeting: "Welcome to CSP! Build projects with Python and algorithms!",
      src: path + "/images/gamify/desert.png",
      pixels: { height: 580, width: 1038 }
    };

    const playerData = {
      id: 'CSP Developer',
      greeting: "I'm building full-stack web applications with Python!",
      src: path + "/images/gamify/chillguy.png",
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: 0, y: height - (height/5) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down: { row: 0, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    const npcData = {
      id: 'CSP Mentor',
      greeting: "Beep! Ready to learn Python and build web applications?",
      src: path + "/images/gamify/r2_idle.png",
      SCALE_FACTOR: 8,
      ANIMATION_RATE: 100,
      pixels: { width: 505, height: 223 },
      INIT_POSITION: { x: width * 1 / 4, y: height * 3 / 4 },
      orientation: { rows: 1, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
      dialogues: [
        "Python is perfect for rapid prototyping and web development!",
        "Learn algorithm efficiency - Big O notation matters!",
        "Flask makes backend development accessible and powerful!",
        "Data structures are the foundation of efficient programs!",
        "SQL databases store and organize your application data!",
        "Team projects teach real-world collaboration skills!"
      ]
    };

    this.classes = [
      { class: GamEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npcData }
    ];
  }
}

export const gameLevelClasses = [GameLevelCSP];
export { GameControl };

In [ ]:
%%html
<!-- UI_RUNNER: CSA Content | panel: zone_csa, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<div id="zone-section-CSA" class="zone-section" style="display: none;">
  <div class="zone-content">
    <h2>☕ CSA Zone - Computer Science A</h2>
    <p><em>Deep dive into Java programming and object-oriented design</em></p>
    
    <h3>CSA - Computer Science A</h3>
    <p class="zone-subtitle-text">Systems</p>
    <p class="zone-description">
      AP ComputerScience A is an in-depth course focusing on programming, algorithms, and data structures. Students gain understanding through analysis, coding, and individual and team projects, establishing fluency in Java.
    </p>
    <h4>Learning Journey:</h4>
    <ul class="zone-journey-list">
      <li>Java OOP Fundamentals</li>
      <li>Classes and Objects</li>
      <li>Inheritance & Polymorphism</li>
      <li>Arrays & ArrayLists</li>
      <li>Recursion Patterns</li>
      <li>Spring Framework Projects</li>
    </ul>
  </div>
</div>

In [ ]:
%%js

// GAME_RUNNER: CSA Zone Game | hide_edit: true, panel: zone_csa, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GamEnvBackground from '/assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1.1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1.1/essentials/Npc.js';
import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';

class GameLevelCSA {
  constructor(gameEnv) {
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const path = gameEnv.path;

    const bgData = {
      name: 'csa-desert',
      greeting: "Welcome to CSA! Master Java and object-oriented programming!",
      src: path + "/images/gamify/desert.png",
      pixels: { height: 580, width: 1038 }
    };

    const playerData = {
      id: 'CSA Engineer',
      greeting: "I'm mastering Java and building enterprise systems!",
      src: path + "/images/gamify/chillguy.png",
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: 0, y: height - (height/5) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down: { row: 0, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    const npcData = {
      id: 'CSA Professor',
      greeting: "Bleep boop! Java is the foundation of enterprise software!",
      src: path + "/images/gamify/r2_idle.png",
      SCALE_FACTOR: 8,
      ANIMATION_RATE: 100,
      pixels: { width: 505, height: 223 },
      INIT_POSITION: { x: width * 1 / 4, y: height * 3 / 4 },
      orientation: { rows: 1, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
      dialogues: [
        "Java's strong typing prevents many runtime errors!",
        "Inheritance creates hierarchies of related classes!",
        "Polymorphism allows flexible, reusable code!",
        "ArrayLists provide dynamic data storage!",
        "Recursion elegantly solves complex problems!",
        "Spring Framework powers modern enterprise applications!"
      ]
    };

    this.classes = [
      { class: GamEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npcData }
    ];
  }
}

export const gameLevelClasses = [GameLevelCSA];
export { GameControl };

In [ ]:
%%html
<!-- UI_RUNNER: CSH Content | panel: zone_csh, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<div id="zone-section-CSH" class="zone-section" style="display: none;">
  <div class="zone-content">
    <h2>🎓 CSH Zone - Computer Science Honors</h2>
    <p><em>Capstone experience with real-world problem solving</em></p>
    
    <h3>CSH - Computer Science Honors</h3>
    <p class="zone-subtitle-text">Capstone</p>
    <p class="zone-description">
      Computer Science "H" is a year-long, senior-only, interdisciplinary honors course serving as the Pathway Capstone. This course functions as a high school senior thesis and culminating honors experience.
    </p>
    <h4>Learning Journey:</h4>
    <ul class="zone-journey-list">
      <li>Capstone Project Planning</li>
      <li>Research & Problem Definition</li>
      <li>Stakeholder Engagement</li>
      <li>System Design & Prototyping</li>
      <li>Technical Documentation</li>
      <li>Public Presentation</li>
    </ul>
  </div>
</div>

In [ ]:
%%js

// GAME_RUNNER: CSH Zone Game | hide_edit: true, panel: zone_csh, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GamEnvBackground from '/assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1.1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1.1/essentials/Npc.js';
import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';

class GameLevelCSH {
  constructor(gameEnv) {
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const path = gameEnv.path;

    const bgData = {
      name: 'csh-desert',
      greeting: "Welcome to CSH! Create your capstone project and make an impact!",
      src: path + "/images/gamify/desert.png",
      pixels: { height: 580, width: 1038 }
    };

    const playerData = {
      id: 'CSH Senior',
      greeting: "I'm creating solutions for real-world problems!",
      src: path + "/images/gamify/chillguy.png",
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: 0, y: height - (height/5) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down: { row: 0, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    const npcData = {
      id: 'CSH Advisor',
      greeting: "Whirrrr! Ready to solve complex problems and present solutions?",
      src: path + "/images/gamify/r2_idle.png",
      SCALE_FACTOR: 8,
      ANIMATION_RATE: 100,
      pixels: { width: 505, height: 223 },
      INIT_POSITION: { x: width * 1 / 4, y: height * 3 / 4 },
      orientation: { rows: 1, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
      dialogues: [
        "Your capstone project showcases all your skills!",
        "Real stakeholders need real solutions!",
        "Research deeply before you design!",
        "Documentation ensures your work lives on!",
        "Presenting to experts builds confidence!",
        "Your thesis contributes to the field!"
      ]
    };

    this.classes = [
      { class: GamEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npcData }
    ];
  }
}

export const gameLevelClasses = [GameLevelCSH];
export { GameControl };

---

**Legacy Navigation:** [Mario Fog-of-War Home]({{ site.baseurl }}/home2)
